# Retrieval v2 – Stage 4 Sparse + Dense Hybrid Retrieval

Stage 4 reuses the frozen L2 BM25 and D1 E5-large window-score caches. It evaluates sparse-only, dense-only, RRF, and three predefined normalized score-fusion settings. Video aggregation remains frozen to P0 max.


## 1. Setup


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount = False)
except Exception :
    pass

DEFAULT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/"
    "aic26/asr_model_comparison"
)
ROOT_OVERRIDE = os.environ.get("AIC_RETRIEVAL_ROOT", "").strip()

ROOT = Path(ROOT_OVERRIDE).expanduser().resolve() if ROOT_OVERRIDE else DEFAULT_ROOT.resolve()
CODE_ROOT = ROOT
ARTIFACT_ROOT = ROOT

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)

RUNNER = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"
CONFIG = CODE_ROOT / "configs" / "retrieval_v2.json"

print("Code root    :", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Runner exists:", RUNNER.exists())
print("Config exists:", CONFIG.exists())

assert RUNNER.exists()
assert CONFIG.exists()


Mounted at /content/drive
Code root    : /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Runner exists: True
Config exists: True


## 2. Frozen Stage 4 contract


In [2]:
config = json.loads(CONFIG.read_text(encoding = "utf-8"))
stage = config["stage04_hybrid"]

print("Retrieval ID:", config["retrieval_id"])
print("Query set   :", stage["query_set"])
print("Corpus      :", stage["corpus"])
print("Sparse      :", config["selection"]["stage01_decision"]["lexical_default"])
print("Dense       :", config["selection"]["stage02_decision"]["dense_default"])
print("Aggregation :", config["selection"]["stage03_decision"]["selected_aggregation"])

print("\nMethods:")
for method_id, item in stage["methods"].items() :
    print(f"  {method_id:20s} {item}")

print("\nStage 4 decision:", config["selection"]["stage04_decision"])


Retrieval ID: aic2026_retrieval_v2_stage00_stage05
Query set   : development20
Corpus      : all50
Sparse      : L2_bm25_preserving
Dense       : D1_e5_large_instruct
Aggregation : P0_max

Methods:
  H0_sparse_only       {'kind': 'identity', 'source': 'sparse'}
  H1_dense_only        {'kind': 'identity', 'source': 'dense'}
  H2_rrf_k60           {'kind': 'rrf', 'k': 60, 'positive_evidence_only': True}
  H3_norm_25_75        {'kind': 'normalized_weighted', 'sparse_weight': 0.25, 'dense_weight': 0.75}
  H4_norm_50_50        {'kind': 'normalized_weighted', 'sparse_weight': 0.5, 'dense_weight': 0.5}
  H5_norm_75_25        {'kind': 'normalized_weighted', 'sparse_weight': 0.75, 'dense_weight': 0.25}

Stage 4 decision: {'status': 'pending_review', 'selected_method': None, 'retained_alternative': None, 'note': 'Run and review Stage 4 before freezing exactly one retrieval architecture.'}


## 3. Execute Stage 4


In [3]:
STAGE = "stage04_hybrid"

command = [
    sys.executable,
    "-u",
    str(RUNNER),
    "--stage",
    STAGE,
]

print("Running:", " ".join(command), "\n")

process = subprocess.Popen(
    command,
    cwd = str(CODE_ROOT),
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
    text = True,
    bufsize = 1,
)

assert process.stdout is not None

for line in process.stdout :
    print(line, end = "")

returncode = process.wait()

if (returncode != 0) :
    raise RuntimeError(f"Retrieval v2 failed with exit code {returncode}")


Running: /usr/bin/python3 -u /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py --stage stage04_hybrid 

AIC 2026 RETRIEVAL V2
Stage:         stage04_hybrid
Code root:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Reports:       /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage04_hybrid

[1/6] Validating Stage 4 contract and frozen Stage 2/3 decisions...
      done in 5.6s
[2/6] Validating development20, All50, and Stage 0-3 prerequisites...
      done in 4.7s
[3/6] Loading frozen L2/D1 score caches and evaluating H0-H5...
/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py:2866: Fu

## 4. Load Stage 4 reports


In [4]:
REPORT_ROOT = ARTIFACT_ROOT / "reports" / "retrieval_v2" / "stage04_hybrid"

summary = json.loads((REPORT_ROOT / "stage_summary.json").read_text(encoding = "utf-8"))
validation = json.loads((REPORT_ROOT / "validation_summary.json").read_text(encoding = "utf-8"))
metrics = pd.read_csv(REPORT_ROOT / "retrieval_metrics.csv")
method_summary = pd.read_csv(REPORT_ROOT / "method_summary.csv")
comparisons = pd.read_csv(REPORT_ROOT / "query_comparison.csv")
diagnostics = pd.read_csv(REPORT_ROOT / "fusion_diagnostics.csv")
latency = pd.read_csv(REPORT_ROOT / "latency_summary.csv")

print("Status :", "PASS" if summary["passed"] else "FAIL")
print("Reports:", REPORT_ROOT)
print("Control reproduction:", summary["control_reproduction_passed"])
print("Source compatibility:", summary["source_compatibility_passed"])


Status : PASS
Reports: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage04_hybrid
Control reproduction: True
Source compatibility: True


## 5. H0–H5 retrieval quality


In [5]:
display(
    metrics[
        [
            "method_id", "model_id", "view",
            "video_recall_at_1", "video_recall_at_5", "video_recall_at_10", "video_mrr",
            "story_recall_at_1", "story_mrr",
        ]
    ].sort_values(["method_id", "model_id", "view"], kind = "mergesort")
)

display(method_summary)


,method_id,model_id,view,video_recall_at_1,video_recall_at_5,video_recall_at_10,video_mrr,story_recall_at_1,story_mrr
0,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,0.45,0.65,0.80,0.571989,0.75,0.795989
1,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,raw,0.45,0.65,0.80,0.571989,0.75,0.795989
2,H0_sparse_only,whisper_large_v3,processed,0.50,0.80,0.90,0.622708,0.70,0.776234
3,H0_sparse_only,whisper_large_v3,raw,0.55,0.85,0.90,0.658864,0.70,0.767522
4,H1_dense_only,parakeet_ctc_0_6b_vietnamese,processed,0.60,0.75,0.85,0.680781,0.75,0.825000
5,H1_dense_only,parakeet_ctc_0_6b_vietnamese,raw,0.65,0.75,0.90,0.726452,0.70,0.833333
6,H1_dense_only,whisper_large_v3,processed,0.65,0.85,0.95,0.753241,0.75,0.828810
7,H1_dense_only,whisper_large_v3,raw,0.70,0.85,0.95,0.760084,0.85,0.877917
8,H2_rrf_k60,parakeet_ctc_0_6b_vietnamese,processed,0.60,0.85,0.90,0.713465,0.75,0.815833
9,H2_rrf_k60,parakeet_ctc_0_6b_vietnamese,raw,0.60,0.85,0.90,0.730333,0.75,0.818810


,method_id,query_count,composite_video_mrr,composite_video_recall_at_1,composite_video_recall_at_5,composite_video_recall_at_10,composite_story_mrr,composite_story_recall_at_1,video_rr_delta_mean_vs_sparse,video_rr_delta_median_vs_sparse,...,video_rr_bootstrap_90_low_vs_sparse,video_rr_bootstrap_90_high_vs_sparse,video_rr_delta_mean_vs_dense,video_rr_delta_median_vs_dense,story_rr_delta_mean_vs_dense,video_better_vs_dense,video_tie_vs_dense,video_worse_vs_dense,video_rr_bootstrap_90_low_vs_dense,video_rr_bootstrap_90_high_vs_dense
0,H0_sparse_only,20,0.606387,0.4875,0.7375,0.8500,0.783933,0.7250,0.000000,0.000000,...,0.000000,0.000000,-0.123752,-0.01172,-0.057332,4,6,10,-0.271443,0.019021
1,H1_dense_only,20,0.730139,0.6500,0.8000,0.9125,0.841265,0.7625,0.123752,0.011720,...,-0.019021,0.271443,0.000000,0.00000,0.000000,0,20,0,0.000000,0.000000
2,H2_rrf_k60,20,0.722477,0.6125,0.8500,0.9000,0.834182,0.7750,0.116089,0.003747,...,-0.008224,0.241634,-0.007663,0.00000,-0.007083,7,7,6,-0.095264,0.072525
3,H3_norm_25_75,20,0.784068,0.7125,0.8375,0.9000,0.877714,0.8250,0.177680,0.008064,...,0.046134,0.315471,0.053928,0.00000,0.036449,6,8,6,0.001424,0.110617
4,H4_norm_50_50,20,0.748188,0.6500,0.8750,0.9000,0.801384,0.7375,0.141801,0.002285,...,0.054883,0.240216,0.018049,0.00000,-0.039881,6,7,7,-0.092492,0.131832
5,H5_norm_75_25,20,0.670029,0.5500,0.8750,0.8750,0.797649,0.7375,0.063641,0.000473,...,0.017777,0.119849,-0.060110,0.00000,-0.043616,5,7,8,-0.193926,0.075426


## 6. Paired comparison against sparse and dense controls


In [6]:
hybrid_vs_dense = comparisons[
    comparisons["reference_method"] == "H1_dense_only"
].copy()

display(
    hybrid_vs_dense.sort_values(
        "video_rr_delta",
        ascending = False,
        kind = "mergesort",
    ).head(30)
)

print("\nLargest regressions vs dense:")
display(
    hybrid_vs_dense.sort_values(
        "video_rr_delta",
        ascending = True,
        kind = "mergesort",
    ).head(30)
)


,reference_method,candidate_method,model_id,view,query_id,reference_story_rank,candidate_story_rank,story_relation,story_rr_delta,reference_video_rank,...,video_relation,video_rr_delta,reference_top_story_window_id,candidate_top_story_window_id,top_story_changed,reference_top_video_id,candidate_top_video_id,top_video_changed,large_regression,top1_catastrophe
468,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,R3-2,3,1,better,0.666667,13,...,better,0.923077,L29_V020_0022,L29_V020_0014,True,K01_V009,L29_V020,True,False,False
708,H1_dense_only,H4_norm_50_50,parakeet_ctc_0_6b_vietnamese,processed,R3-2,3,1,better,0.666667,13,...,better,0.923077,L29_V020_0022,L29_V020_0014,True,K01_V009,L29_V020,True,False,False
788,H1_dense_only,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,processed,R3-2,3,1,better,0.666667,13,...,better,0.923077,L29_V020_0022,L29_V020_0014,True,K01_V009,L29_V020,True,False,False
448,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,raw,R3-2,2,1,better,0.500000,9,...,better,0.888889,L29_V020_0022,L29_V020_0014,True,K01_V009,L29_V020,True,False,False
688,H1_dense_only,H4_norm_50_50,parakeet_ctc_0_6b_vietnamese,raw,R3-2,2,1,better,0.500000,9,...,better,0.888889,L29_V020_0022,L29_V020_0015,True,K01_V009,L29_V020,True,False,False
768,H1_dense_only,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,raw,R3-2,2,1,better,0.500000,9,...,better,0.888889,L29_V020_0022,L29_V020_0014,True,K01_V009,L29_V020,True,False,False
408,H1_dense_only,H0_sparse_only,whisper_large_v3,raw,R3-2,10,1,better,0.900000,7,...,better,0.857143,L29_V020_0008,L29_V020_0014,True,L30_V095,L29_V020,True,False,False
648,H1_dense_only,H4_norm_50_50,whisper_large_v3,raw,R3-2,10,1,better,0.900000,7,...,better,0.857143,L29_V020_0008,L29_V020_0014,True,L30_V095,L29_V020,True,False,False
728,H1_dense_only,H5_norm_75_25,whisper_large_v3,raw,R3-2,10,1,better,0.900000,7,...,better,0.857143,L29_V020_0008,L29_V020_0014,True,L30_V095,L29_V020,True,False,False
428,H1_dense_only,H0_sparse_only,whisper_large_v3,processed,R3-2,7,1,better,0.857143,4,...,better,0.750000,L29_V020_0022,L29_V020_0014,True,L30_V057,L29_V020,True,False,False



Largest regressions vs dense:


,reference_method,candidate_method,model_id,view,query_id,reference_story_rank,candidate_story_rank,story_relation,story_rr_delta,reference_video_rank,...,video_relation,video_rr_delta,reference_top_story_window_id,candidate_top_story_window_id,top_story_changed,reference_top_video_id,candidate_top_video_id,top_video_changed,large_regression,top1_catastrophe
442,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,raw,R1-13,2,3,worse,-0.166667,1,...,worse,-0.960000,L30_V095_0004,L30_V095_0005,True,L30_V095,L21_V015,True,True,True
462,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,R1-13,4,3,better,0.083333,1,...,worse,-0.960000,L30_V095_0004,L30_V095_0005,True,L30_V095,L21_V015,True,True,True
762,H1_dense_only,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,raw,R1-13,2,3,worse,-0.166667,1,...,worse,-0.937500,L30_V095_0004,L30_V095_0005,True,L30_V095,L21_V015,True,True,False
782,H1_dense_only,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,processed,R1-13,4,3,better,0.083333,1,...,worse,-0.923077,L30_V095_0004,L30_V095_0005,True,L30_V095,L21_V015,True,True,False
459,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,raw,R3-28,1,1,tie,0.000000,1,...,worse,-0.916667,K12_V007_0015,K12_V007_0015,False,K12_V007,L25_V058,True,True,False
479,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,R3-28,1,1,tie,0.000000,1,...,worse,-0.916667,K12_V007_0015,K12_V007_0015,False,K12_V007,L25_V058,True,True,False
425,H1_dense_only,H0_sparse_only,whisper_large_v3,processed,R1-22,1,2,worse,-0.500000,1,...,worse,-0.888889,L26_V178_0002,L26_V178_0006,True,L26_V178,L26_V222,True,False,False
445,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,raw,R1-22,1,1,tie,0.000000,1,...,worse,-0.875000,L26_V178_0002,L26_V178_0002,False,L26_V178,K19_V022,True,False,False
465,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,R1-22,1,1,tie,0.000000,1,...,worse,-0.875000,L26_V178_0002,L26_V178_0002,False,L26_V178,K19_V022,True,False,False
474,H1_dense_only,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,R2-20,1,1,tie,0.000000,1,...,worse,-0.875000,K06_V010_0005,K06_V010_0005,False,K06_V010,K03_V023,True,False,False


## 7. Fusion diagnostics and catastrophic failures


In [7]:
catastrophic = diagnostics[
    diagnostics["large_regression_vs_dense"].astype(bool)
    | diagnostics["top1_catastrophe_vs_dense"].astype(bool)
].copy()

print("Catastrophic / large dense-reference regressions:", len(catastrophic))
display(
    catastrophic.sort_values(
        ["top1_catastrophe_vs_dense", "video_rank_delta_vs_dense"],
        ascending = [False, False],
        kind = "mergesort",
    )
)

print("\nCached-score Stage 4 timing only:")
display(latency)


Catastrophic / large dense-reference regressions: 20


,method_id,model_id,view,query_id,sparse_story_rank,dense_story_rank,hybrid_story_rank,sparse_video_rank,dense_video_rank,hybrid_video_rank,...,top1_catastrophe_vs_sparse,top1_catastrophe_vs_dense,sparse_min,sparse_max,sparse_range,sparse_constant,dense_min,dense_max,dense_range,dense_constant
250,H5_norm_75_25,whisper_large_v3,raw,R2-1,12,1,6,33,6,35,...,False,False,0.0,26.302969,26.302969,False,0.737979,0.869820,0.131841,False
270,H5_norm_75_25,whisper_large_v3,processed,R2-1,11,1,6,34,9,32,...,False,False,0.0,26.301542,26.301542,False,0.732763,0.863711,0.130948,False
290,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,raw,R2-1,13,1,6,34,9,32,...,False,False,0.0,25.166786,25.166786,False,0.743489,0.873872,0.130383,False
170,H4_norm_50_50,whisper_large_v3,raw,R2-1,12,1,4,33,6,27,...,False,False,0.0,26.302969,26.302969,False,0.737979,0.869820,0.131841,False
210,H4_norm_50_50,parakeet_ctc_0_6b_vietnamese,raw,R2-1,13,1,4,34,9,27,...,False,False,0.0,25.166786,25.166786,False,0.743489,0.873872,0.130383,False
296,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,raw,R2-8,7,2,6,27,11,28,...,False,False,0.0,39.552456,39.552456,False,0.734365,0.885085,0.150720,False
50,H2_rrf_k60,parakeet_ctc_0_6b_vietnamese,raw,R2-1,13,1,1,34,9,25,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,H5_norm_75_25,parakeet_ctc_0_6b_vietnamese,processed,R2-1,13,1,7,34,16,32,...,False,False,0.0,25.162140,25.162140,False,0.764099,0.869365,0.105266,False
16,H2_rrf_k60,whisper_large_v3,raw,R2-8,11,1,1,33,17,32,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,H3_norm_25_75,whisper_large_v3,raw,R2-8,11,1,1,33,17,32,...,False,False,0.0,37.611916,37.611916,False,0.726259,0.877333,0.151074,False



Cached-score Stage 4 timing only:


,method_id,model_id,view,query_count,fusion_total_ms,fusion_per_query_ms,aggregation_and_ranking_total_ms,aggregation_and_ranking_per_query_ms,cached_score_stage_total_ms,cached_score_stage_per_query_ms
0,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,processed,20,0.121467,0.006073,195.564256,9.778213,195.685723,9.784286
1,H0_sparse_only,parakeet_ctc_0_6b_vietnamese,raw,20,0.129939,0.006497,193.787163,9.689358,193.917102,9.695855
2,H0_sparse_only,whisper_large_v3,processed,20,0.091478,0.004574,181.333443,9.066672,181.424921,9.071246
3,H0_sparse_only,whisper_large_v3,raw,20,0.186167,0.009308,313.022837,15.651142,313.209004,15.660450
4,H1_dense_only,parakeet_ctc_0_6b_vietnamese,processed,20,0.102141,0.005107,205.661400,10.283070,205.763541,10.288177
5,H1_dense_only,parakeet_ctc_0_6b_vietnamese,raw,20,0.097871,0.004894,187.358760,9.367938,187.456631,9.372832
6,H1_dense_only,whisper_large_v3,processed,20,0.096033,0.004802,188.143472,9.407174,188.239505,9.411975
7,H1_dense_only,whisper_large_v3,raw,20,0.171122,0.008556,214.379290,10.718965,214.550412,10.727521
8,H2_rrf_k60,parakeet_ctc_0_6b_vietnamese,processed,20,96.941122,4.847056,197.364735,9.868237,294.305857,14.715293
9,H2_rrf_k60,parakeet_ctc_0_6b_vietnamese,raw,20,102.859721,5.142986,199.111156,9.955558,301.970877,15.098544


## 8. Decision

Do not select automatically. Review Video MRR and R@1 first, then R@5/R@10, paired better/tie/worse behavior, Story guardrails, catastrophic failures, and simplicity. After review, freeze exactly one `selection.stage04_decision.selected_method` in `configs/retrieval_v2.json` before running Stage 5.
